# Shapley Values — Explained from Scratch

**Who this is for**: A full-stack engineer who understands code and logic, is learning ML, and wants to understand *why* a model made a specific prediction.

**Author:** Tom McTavish

---

## The problem we're solving

You've trained a model. It predicts that a loan applicant should be **rejected**.

You have four features: age, income, debt, and credit score.

The model says: rejected. But *why*? Was it mainly the debt? The age? Some combination?

This is the **model interpretability** problem. Shapley values are one answer to it.

---

## The core idea in plain English

A Shapley value answers this question:

> **"How much did this one feature contribute to pushing this prediction above (or below) the average prediction?"**

Not globally across all data. Not for the model in general. For **this specific prediction**, for **this specific person**.

Each feature gets a number — its Shapley value. Positive means it pushed the prediction up. Negative means it pushed it down. And they all add up to the total difference between this prediction and the average prediction.

---

## A concrete analogy before any code

Imagine your team ships a feature. Revenue goes up by $10,000. Four engineers worked on it: Alice, Bob, Carol, Dave.

**Question**: How do you fairly credit each engineer?

You can't just say "it's equal" — Alice did the backend, Bob wrote one CSS line. You can't just look at who worked *last*, because order shouldn't matter.

The Shapley approach: **try every possible order in which the team could have assembled, and for each order, measure how much revenue appeared when each person joined.**

- Order: Alice → Bob → Carol → Dave. When Alice joined alone: +$6k. When Bob joined: +$1k. When Carol joined: +$2k. When Dave joined: +$1k.
- Order: Dave → Alice → Carol → Bob. When Dave joined alone: +$1k. When Alice joined: +$7k. Etc.

Average each person's contribution across **all possible orderings**. That average is their Shapley value. It's the fairest possible attribution.

**In ML**: replace "engineers" with "features" and "revenue" with "model prediction."

---

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris

shap.initjs()

---

## Step 1 — Train a simple model

We'll use the Iris dataset. It's a classic: 150 flowers, 4 measurements each, classified into 3 species.

The 4 features are:
- sepal length
- sepal width
- petal length
- petal width

Think of features as the "inputs" to your model — like columns in a database row.

In [ ]:
iris = load_iris()
X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train)

print(f'Model accuracy: {clf.score(X_test, y_test):.1%}')
print()
print('A sample flower we will explain:')
sample = X_test.iloc[[0]]
display(sample)
proba = clf.predict_proba(sample)[0]
print(f'Model predicts: {iris.target_names[proba.argmax()]} ({proba.max():.1%} confident)')

---

## Step 2 — What's the "average prediction"?

Before we can say a feature pushed the prediction *up* or *down*, we need a reference point: **what would the model predict if it knew nothing about this specific flower?**

The answer: the average prediction across all training flowers.

This is sometimes called the **baseline** or **expected value**. It's the model's "default" before it sees any of your features.

In [ ]:
# Average probability the model assigns to each class across all training data
avg_proba = clf.predict_proba(X_train).mean(axis=0)

print('Average (baseline) prediction probabilities across training set:')
for species, p in zip(iris.target_names, avg_proba):
    print(f'  {species}: {p:.3f}')

print()
print('Our sample flower prediction probabilities:')
for species, p in zip(iris.target_names, proba):
    print(f'  {species}: {p:.3f}')

print()
print('Difference (what the Shapley values need to explain):')
for species, diff in zip(iris.target_names, proba - avg_proba):
    print(f'  {species}: {diff:+.3f}')

---

## Step 3 — Computing Shapley values

To compute Shapley values exactly, you'd need to try every possible subset of features and measure how the prediction changes when you add each feature to that subset.

With 4 features there are 2⁴ = 16 subsets — manageable. With 20 features it's over a million. With 50 it's impossible.

For **tree-based models** (like our Random Forest), there's a smarter algorithm — `TreeExplainer` — that computes exact Shapley values by walking the trees directly, without trying every subset. It's fast and exact.

In [ ]:
explainer = shap.TreeExplainer(clf)
shap_values = explainer(X_test)

# shap_values.values has shape: (n_test_samples, n_features, n_classes)
print('Shape of shap_values.values:', shap_values.values.shape)
print(f'  → {shap_values.values.shape[0]} test samples')
print(f'  → {shap_values.values.shape[1]} features')
print(f'  → {shap_values.values.shape[2]} classes')

In [ ]:
# Look at the Shapley values for our sample flower, for each class
sample_shap = shap_values.values[0]  # shape: (4 features, 3 classes)

print('Shapley values for our sample flower:')
df_shap = pd.DataFrame(
    sample_shap,
    index=iris.feature_names,
    columns=[f'φ ({c})' for c in iris.target_names]
)
display(df_shap.round(4))

print()
print('Each column sums to (prediction - baseline) for that class:')
for i, species in enumerate(iris.target_names):
    total = sample_shap[:, i].sum()
    diff = proba[i] - avg_proba[i]
    print(f'  {species}: sum of φ = {total:+.4f},  prediction - baseline = {diff:+.4f}')

**What you're seeing**: the Shapley values for each feature tell you exactly how much that feature moved the prediction away from the baseline — for this one flower.

And they sum to the total movement. Nothing is lost, nothing is invented. This is called the **efficiency** property.

---

## Step 4 — Visualizing a single prediction: the Waterfall plot

The waterfall plot is the clearest way to read a single prediction.

How to read it:
- **Bottom**: starts at the baseline `E[f(X)]` — the average prediction
- **Each bar**: one feature's Shapley value — how much it pushed the prediction up (red) or down (blue)
- **Top**: arrives at the final prediction `f(x)`

Think of it like a bank statement: starting balance → each transaction → ending balance.

In [ ]:
# Build an Explanation object for class 0 (setosa)
class_idx = 0
exp = shap.Explanation(
    values=shap_values.values[:, :, class_idx],
    base_values=shap_values.base_values[:, class_idx],
    data=X_test.values,
    feature_names=iris.feature_names
)

print(f'Waterfall plot for our sample flower — class: {iris.target_names[class_idx]}')
shap.plots.waterfall(exp[0])

---

## Step 5 — Visualizing many predictions at once: the Summary plot

The waterfall shows one flower. The summary plot shows all test flowers at once.

How to read it:
- **Y-axis**: features, ranked by how important they are overall (biggest average impact at top)
- **X-axis**: the Shapley value — positive means "pushed prediction up", negative means "pushed it down"
- **Each dot**: one flower from the test set
- **Color**: the actual value of that feature — red = high value, blue = low value

So if you see *red dots on the right* for petal length, that means: flowers with long petals got their prediction pushed *up* for this class.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for class_idx, class_name in enumerate(iris.target_names):
    plt.sca(axes[class_idx])
    shap.summary_plot(
        shap_values.values[:, :, class_idx],
        X_test,
        show=False,
        plot_size=None
    )
    axes[class_idx].set_title(f'Class: {class_name}', fontsize=13)

plt.tight_layout()
plt.show()

---

## Step 6 — Which features matter most overall? The Bar plot

Sometimes you just want a simple ranking: which features had the biggest impact on predictions, on average?

The bar plot shows the **mean absolute Shapley value** per feature — the average size of a feature's impact, ignoring direction.

In [ ]:
# Mean absolute Shapley value per feature, for class 0
mean_abs = np.abs(shap_values.values[:, :, 0]).mean(axis=0)
feature_importance = pd.Series(mean_abs, index=iris.feature_names).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 4))
feature_importance.plot.barh(ax=ax, color='steelblue')
ax.set_xlabel('Mean |Shapley value| (impact on prediction)')
ax.set_title('Feature importance for class: setosa')
ax.axvline(0, color='black', lw=0.8)
plt.tight_layout()
plt.show()

---

## Step 7 — Build the intuition manually

Let's implement a tiny version of the Shapley algorithm by hand. This is the best way to really understand what's happening.

**The recipe:**

1. Pick a feature `j` you want to explain.
2. Pick a random ordering of all features.
3. Build two "hybrid" flower instances:
   - One where features up-to-and-including `j` (in this ordering) come from our flower, rest from a random background flower.
   - One where features up-to-but-not-including `j` come from our flower, rest from background.
4. The difference in predictions between those two hybrids is `j`'s contribution in this ordering.
5. Repeat across many random orderings and average.

That's it. The "Frankenstein" hybrid instances are how we isolate each feature's contribution.

In [ ]:
def estimate_shapley_values(model, instance, background, n_samples=500, class_idx=0, random_state=42):
    """
    Monte Carlo estimate of Shapley values for one instance.

    For each random ordering and each feature j:
      - 'with_j' hybrid: features before j AND j come from instance, rest from background
      - 'without_j' hybrid: features before j come from instance, j AND rest from background
    Marginal contribution = predict(with_j) - predict(without_j)
    Average over many orderings.
    """
    rng = np.random.default_rng(random_state)
    n_features = len(instance)
    phi = np.zeros(n_features)

    for _ in range(n_samples):
        # Random background row and random feature ordering
        bg = background[rng.integers(len(background))].copy()
        ordering = rng.permutation(n_features)

        for pos, j in enumerate(ordering):
            # Features at positions 0..pos (inclusive) → from instance
            with_j = bg.copy()
            for k in ordering[:pos + 1]:
                with_j[k] = instance[k]

            # Features at positions 0..pos-1 → from instance (j stays as background)
            without_j = bg.copy()
            for k in ordering[:pos]:
                without_j[k] = instance[k]

            pred_with = model.predict_proba(with_j.reshape(1, -1))[0, class_idx]
            pred_without = model.predict_proba(without_j.reshape(1, -1))[0, class_idx]

            phi[j] += pred_with - pred_without

    return phi / n_samples


instance = X_test.values[0]
background = X_train.values

phi_manual = estimate_shapley_values(clf, instance, background, n_samples=500, class_idx=0)
phi_shap = shap_values.values[0, :, 0]  # From the TreeExplainer

print('Comparison — manual Monte Carlo vs TreeExplainer (class: setosa):')
print(f'{"Feature":<25} {"Manual":>10} {"TreeExplainer":>14}')
print('-' * 52)
for name, manual, exact in zip(iris.feature_names, phi_manual, phi_shap):
    print(f'{name:<25} {manual:>10.4f} {exact:>14.4f}')

The manual estimates are close to the exact values from TreeExplainer — the difference is just noise from Monte Carlo sampling. More samples → closer.

This confirms you now understand what TreeExplainer is actually computing under the hood.

---

## Step 8 — Three things that will trip you up

### Trap 1: A Shapley value is NOT "what happens if I remove this feature"

This is the most common misreading.

If petal length has a Shapley value of `+0.20`, it does NOT mean: "if I deleted petal length from the input, the prediction would drop by 0.20."

It means: "across all possible combinations of features, petal length's presence contributed +0.20 on average."

The difference matters because removing a feature entirely is a different operation from the Frankenstein hybrid approach Shapley uses.

---

### Trap 2: Shapley values are not causal

If income has a high Shapley value, it doesn't mean income *caused* the outcome. It means the model used income heavily for this prediction. The model could be wrong. Income could be a proxy for something else.

Shapley explains the model. It doesn't explain the world.

---

### Trap 3: Correlated features distort the picture

The Frankenstein hybrid approach assumes you can freely mix features from different flowers. But in real data, features are often correlated — tall people tend to weigh more; people with high income tend to have less debt.

When you mix features naively, you create impossible combinations (short + heavy, high income + huge debt). The model has never seen these and its predictions on them can be misleading.

TreeExplainer has an option `feature_perturbation='tree_path_dependent'` that handles this better for tree models.

In [ ]:
# Demonstrate Trap 3: same model, different Shapley values depending on reference data
explainer_full = shap.TreeExplainer(clf, X_train)  # background = full training set
explainer_small = shap.TreeExplainer(clf, X_train.iloc[:10])  # background = 10 samples

sv_full = explainer_full(X_test.iloc[[0]])
sv_small = explainer_small(X_test.iloc[[0]])

print('Same model, same prediction, different background data → different Shapley values')
print(f'(class: setosa, feature: petal length)')
print(f'  Full background:  {sv_full.values[0, 2, 0]:+.4f}')
print(f'  10-sample background: {sv_small.values[0, 2, 0]:+.4f}')
print()
print('This is why your choice of reference data matters.')

---

## Summary

| Question | Answer |
|---|---|
| What is a Shapley value? | How much one feature contributed to pushing *this* prediction above or below the average |
| How is it computed? | Average the feature's marginal contribution across all possible orderings of features |
| Why average across orderings? | So the result doesn't depend on an arbitrary order — it's the fairest split possible |
| Do Shapley values sum to anything? | Yes — they sum to `prediction − average_prediction` (efficiency property) |
| Why not compute exactly? | 2^n subsets needed — exponential. Use TreeExplainer for trees (fast + exact) |
| What does a positive Shapley value mean? | This feature pushed *this prediction* above the baseline |
| Does that mean increasing the feature increases the prediction? | **No.** Shapley is about the current value, not about changing it |
| Is it causal? | **No.** It explains the model, not the real world |

---

## What to read next

- [Interpretable ML Book — Shapley Values](https://christophm.github.io/interpretable-ml-book/shapley.html) — the source for this notebook
- [SHAP library docs](https://shap.readthedocs.io/) — API reference and more plot types
- [Original SHAP paper (Lundberg & Lee, 2017)](https://proceedings.neurips.cc/paper_files/paper/2017/file/8a20a8621978632d76c43dfd28b67767-Paper.pdf)